# Phase 1 — Test Search & Dedup Tools

End-to-end validation of `search_pubmed`, `search_semantic_scholar`, and `dedupe_papers`.

Test query: *wearable AND benchmark AND deep learning AND validation*

**Note:** Semantic Scholar may return empty results without an API key due to aggressive rate limiting. PubMed works reliably. Both functions fail gracefully (return `[]`) on 429s.

In [ ]:
import sys, logging, time
sys.path.insert(0, "..")
logging.basicConfig(level=logging.INFO)

from lit_review_agent.tools import search_pubmed, search_semantic_scholar, dedupe_papers
from lit_review_agent.state import Paper
print("Imports OK")

## 1. PubMed Search

In [ ]:
query = '(wearable) AND (benchmark OR validation) AND (deep learning OR machine learning)'
pubmed_papers = search_pubmed(query, max_results=10)

print(f"PubMed returned {len(pubmed_papers)} papers\n")
for p in pubmed_papers[:3]:
    print(f"  [{p.year}] {p.title[:80]}...")
    print(f"    DOI={p.doi}  PMID={p.pmid}")
    print(f"    Authors: {', '.join(p.authors[:3])}{'...' if len(p.authors) > 3 else ''}")
    print(f"    Abstract: {p.abstract[:120]}...\n")

# Pause before next API call to respect rate limits
time.sleep(1)

## 2. Semantic Scholar Search

In [ ]:
# Note: S2 may return [] due to rate limiting without an API key.
# If rate-limited, it will retry with backoff and eventually return empty.
s2_papers = search_semantic_scholar("wearable benchmark deep learning validation", max_results=10)

print(f"Semantic Scholar returned {len(s2_papers)} papers\n")
for p in s2_papers[:3]:
    print(f"  [{p.year}] {p.title[:80]}...")
    print(f"    DOI={p.doi}  PMID={p.pmid}")
    print(f"    Authors: {', '.join(p.authors[:3])}{'...' if len(p.authors) > 3 else ''}")
    print(f"    Abstract: {p.abstract[:120]}...\n")

## 3. Deduplication

Combine results from both sources and deduplicate by DOI / normalized title.

In [ ]:
combined = pubmed_papers + s2_papers
print(f"Combined: {len(combined)} papers ({len(pubmed_papers)} PubMed + {len(s2_papers)} S2)")

unique = dedupe_papers(combined)
print(f"After dedup: {len(unique)} unique papers")

# Show source breakdown
from collections import Counter
source_counts = Counter(p.source for p in unique)
print(f"Sources: {dict(source_counts)}")

## 4. Inspect a Single Paper (full schema)

In [ ]:
# Pick the first paper and show full schema
if unique:
    p = unique[0]
    print(p.model_dump_json(indent=2))